In [1]:
import os
os.makedirs('src', exist_ok=True)
print(os.listdir('.'))

['02_greedy_construction.ipynb', '00_export_src.ipynb', '.ipynb_checkpoints', '03_local_search.ipynb', '01_vrptw_setup.ipynb', 'src']


In [2]:
%%writefile src/vrptw.py
import math
import random


class Customer:
    def __init__(self, id, x, y, demand, ready_time, due_time, service_time=0):
        self.id = id
        self.x = x
        self.y = y
        self.demand = demand
        self.ready_time = ready_time
        self.due_time = due_time
        self.service_time = service_time

    def __repr__(self):
        return f"Customer({self.id}, demand={self.demand}, window=[{self.ready_time},{self.due_time}])"


def distance(c1, c2):
    return math.hypot(c1.x - c2.x, c1.y - c2.y)


class VRPTWInstance:
    def __init__(self, depot, customers, vehicle_capacity, num_vehicles):
        self.depot = depot
        self.customers = customers
        self.vehicle_capacity = vehicle_capacity
        self.num_vehicles = num_vehicles

    def all_nodes(self):
        return [self.depot] + self.customers


def generate_vrptw_instance(n_customers, vehicle_capacity=100, num_vehicles=5, seed=42):
    rng = random.Random(seed)
    depot = Customer("Depot", 50, 50, 0, 0, 1000, 0)
    customers = []
    for i in range(n_customers):
        x, y = rng.randint(0, 100), rng.randint(0, 100)
        demand = rng.randint(5, 30)
        ready = rng.randint(0, 400)
        due = ready + rng.randint(50, 200)
        service = rng.randint(5, 15)
        customers.append(Customer(f"C{i}", x, y, demand, ready, due, service))
    return VRPTWInstance(depot, customers, vehicle_capacity, num_vehicles)


def route_feasible(route, instance):
    total_demand = sum(c.demand for c in route)
    if total_demand > instance.vehicle_capacity:
        return False, None, None
    current_time = 0.0
    current_pos = instance.depot
    total_dist = 0.0
    arrival_times = []
    for customer in route:
        travel = distance(current_pos, customer)
        arrival = current_time + travel
        start_service = max(arrival, customer.ready_time)
        if start_service > customer.due_time:
            return False, None, None
        arrival_times.append(start_service)
        current_time = start_service + customer.service_time
        total_dist += travel
        current_pos = customer
    total_dist += distance(current_pos, instance.depot)
    return True, total_dist, arrival_times


def solution_total_distance(routes, instance):
    total = 0.0
    for route in routes:
        feasible, dist, _ = route_feasible(route, instance)
        if not feasible:
            return None
        total += dist
    return total

Writing src/vrptw.py


In [3]:
%%writefile src/greedy.py
from vrptw import distance


def greedy_construction(instance):
    unvisited = list(instance.customers)
    routes = []
    for _ in range(instance.num_vehicles):
        if not unvisited:
            break
        route = []
        current_pos = instance.depot
        current_time = 0.0
        current_load = 0
        while True:
            best_customer = None
            best_dist = float('inf')
            for customer in unvisited:
                if current_load + customer.demand > instance.vehicle_capacity:
                    continue
                travel = distance(current_pos, customer)
                arrival = current_time + travel
                start_service = max(arrival, customer.ready_time)
                if start_service > customer.due_time:
                    continue
                if travel < best_dist:
                    best_dist = travel
                    best_customer = customer
            if best_customer is None:
                break
            route.append(best_customer)
            travel = distance(current_pos, best_customer)
            arrival = current_time + travel
            current_time = max(arrival, best_customer.ready_time) + best_customer.service_time
            current_load += best_customer.demand
            current_pos = best_customer
            unvisited.remove(best_customer)
        if route:
            routes.append(route)
    return routes, unvisited

Writing src/greedy.py


In [4]:
%%writefile src/local_search.py
from vrptw import route_feasible, solution_total_distance


def two_opt_route(route, instance):
    improved = True
    best_route = route[:]
    _, best_dist, _ = route_feasible(best_route, instance)

    while improved:
        improved = False
        for i in range(len(best_route) - 1):
            for j in range(i + 1, len(best_route)):
                candidate = best_route[:i] + best_route[i:j+1][::-1] + best_route[j+1:]
                feasible, dist, _ = route_feasible(candidate, instance)
                if feasible and dist < best_dist:
                    best_route = candidate
                    best_dist = dist
                    improved = True

    return best_route, best_dist


def inter_route_swap(routes, instance):
    routes = [r[:] for r in routes]
    improved = True

    while improved:
        improved = False
        current_total = solution_total_distance(routes, instance)

        for i, route_i in enumerate(routes):
            for c_idx, customer in enumerate(route_i):
                for j, route_j in enumerate(routes):
                    if i == j:
                        continue
                    for insert_pos in range(len(route_j) + 1):
                        new_route_i = route_i[:c_idx] + route_i[c_idx+1:]
                        new_route_j = route_j[:insert_pos] + [customer] + route_j[insert_pos:]

                        feasible_i, dist_i, _ = route_feasible(new_route_i, instance) if new_route_i else (True, 0, [])
                        feasible_j, dist_j, _ = route_feasible(new_route_j, instance)

                        if feasible_i and feasible_j:
                            trial_routes = routes[:]
                            trial_routes[i] = new_route_i
                            trial_routes[j] = new_route_j
                            trial_routes = [r for r in trial_routes if r]
                            trial_total = solution_total_distance(trial_routes, instance)

                            if trial_total is not None and trial_total < current_total:
                                routes = trial_routes
                                improved = True
                                current_total = trial_total
                                break
                    if improved:
                        break
                if improved:
                    break
            if improved:
                break

    return routes


def local_search(routes, instance):
    routes = [r[:] for r in routes]
    prev_total = solution_total_distance(routes, instance)
    while True:
        new_routes = []
        for route in routes:
            improved_route, _ = two_opt_route(route, instance)
            new_routes.append(improved_route)
        routes = new_routes

        routes = inter_route_swap(routes, instance)

        new_total = solution_total_distance(routes, instance)
        if new_total >= prev_total - 1e-9:
            break
        prev_total = new_total

    return routes

Writing src/local_search.py


In [5]:
import os
print(os.listdir('src'))

['local_search.py', 'greedy.py', 'vrptw.py']
